# Descargar las capas del SIG de Asturias (paso 1: datos *raw*)

**Qué hace:** descarga capas oficiales del servidor SIG del Principado (SITPA-IDEAS) y las guarda
**tal cual**, en GeoJSON y con la fecha de descarga en el nombre. No limpia nada: eso es el paso 2.

**Por qué así:** los servicios cambian o se caen. Guardando una copia fechada, la evaluación del
TFG se puede repetir con los mismos datos y en la memoria se cita "datos descargados el día X".

**Qué descarga** (editable en `SERVICES`):
- `Sanidad/Edificios_Sanitarios`: hospitales, centros de salud, consultorios…
- `Sanidad/Farmacias_Botiquines`: farmacias y botiquines.
- `Visor/Educacion`: centros de enseñanza.
- `UnidadesAdministrativas`: **localidades** (puntos con código INE y habitantes), **parroquias** (SADEI) y límites.
- `Visor/Salud`, solo la capa 10: **zonas básicas de salud** (polígonos).
- `VisorRPGUR/Visor_RPGUR` (planeamiento urbanístico, otra ruta del servidor): **clasificación del suelo**,
  suelo urbanizable, **núcleos rurales**, áreas de prevención de riesgos y planeamiento general vigente.

Las tres primeras ya se descargaron el 11/10/2026. Para no repetirlas, comenta sus líneas en `SERVICES`.

**Cómo funciona:** cada servicio ArcGIS REST tiene una lista de capas (`/layers?f=json`). Para cada
capa se piden primero los identificadores de todos sus elementos y luego los elementos por bloques,
porque el servidor devuelve como máximo unos 2.000 por consulta.

Además del GeoJSON, guarda un `manifest_<fecha>.json` con la URL, la fecha y el número de elementos
de cada capa: es la "ficha" de la descarga para la memoria.

In [1]:
%pip install -q geopandas requests


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import time
from datetime import date

import geopandas as gpd
import requests

# Base URL of the Asturias GIS server (ArcGIS REST).
BASE_URL = "https://sig.asturias.es/servicios/rest/services"

# Services to download: service -> list of layer ids, or None for all its feature layers.
SERVICES = {
    # "Sanidad/Edificios_Sanitarios": None,
    # "Sanidad/Farmacias_Botiquines": None,
    # "Visor/Educacion": None,
    # "UnidadesAdministrativas": None,  # localities (INE code + population), parishes, boundaries
    # "Visor/Salud": [10],              # basic health zones (polygons); other layers repeat Sanidad/*
    # Urban planning lives under another path of the same server, so it goes as a full URL.
    # 41 land classification, 39 developable land, 37 rural settlements,
    # 22 risk prevention areas, 57 general plan in force.
    "https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR": [41, 39, 37, 22, 57],
}

# Output folder. In Colab with Drive mounted, e.g. "/content/drive/MyDrive/tfg/data/raw".
OUTPUT_DIR = "data/raw"

# Download date, used in file names so each download is a frozen snapshot.
TODAY = date.today().isoformat()

# Elements requested per call (below the server limit of ~2,000).
CHUNK_SIZE = 500

HEADERS = {"User-Agent": "TFG-UOC-gis-agent/0.1 (academic project)"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Funciones de descarga

- `get_json`: hace una petición y reintenta si el servidor falla (pasa a menudo con servicios públicos).
- `list_layers`: devuelve las capas de un servicio, sin los "grupos" (carpetas que no tienen datos).
- `download_layer`: descarga todos los elementos de una capa en coordenadas WGS84 (lat/lon, EPSG:4326).

In [3]:
def get_json(url, params=None, retries=3, wait_s=10):
    """GET a URL and return the parsed JSON, retrying on network or server errors."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=60)
            response.raise_for_status()
            data = response.json()
            # ArcGIS reports some errors with HTTP 200 and an "error" key in the body.
            if isinstance(data, dict) and "error" in data:
                raise RuntimeError(data["error"])
            return data
        except Exception as error:
            print(f"   intento {attempt}/{retries} fallido: {error}")
            if attempt == retries:
                raise
            time.sleep(wait_s)


def service_url(service):
    """Return the REST URL of a service, given as a path under BASE_URL or as a full URL."""
    return service if service.startswith("https://") else f"{BASE_URL}/{service}"


def list_layers(service):
    """Return the feature layers (id, name, geometry type) of a MapServer service."""
    info = get_json(f"{service_url(service)}/MapServer/layers", {"f": "json"})
    return [
        {"id": layer["id"], "name": layer["name"], "geometry": layer.get("geometryType")}
        for layer in info["layers"]
        if layer.get("type") == "Feature Layer"  # skip group layers (no data of their own)
    ]


def download_layer(service, layer_id):
    """Download every feature of a layer as a GeoJSON dict (EPSG:4326)."""
    query_url = f"{service_url(service)}/MapServer/{layer_id}/query"
    # 1) Ask only for the ids: this call is not limited by the server's max record count.
    ids = get_json(query_url, {"where": "1=1", "returnIdsOnly": "true", "f": "json"})
    object_ids = sorted(ids.get("objectIds") or [])
    features = []
    # 2) Fetch the features in chunks of ids.
    for start in range(0, len(object_ids), CHUNK_SIZE):
        chunk = object_ids[start:start + CHUNK_SIZE]
        page = get_json(query_url, {
            "objectIds": ",".join(map(str, chunk)),
            "outFields": "*",
            "outSR": 4326,
            "f": "geojson",
        })
        features.extend(page.get("features", []))
    return {"type": "FeatureCollection", "features": features}, len(object_ids)


def slug(text):
    """Turn a layer name into a safe file name: 'Centros de Salud' -> 'centros_de_salud'."""
    text = text.lower()
    for accented, plain in zip("áéíóúüñ", "aeiouun"):
        text = text.replace(accented, plain)
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

## 2. Ver qué capas tiene cada servicio

Antes de descargar, imprime las capas. Si algún servicio falla aquí, el servidor no es accesible
desde donde ejecutas el cuaderno (anótalo: es un riesgo de datos).

In [4]:
layers_by_service = {}
for service, wanted_ids in SERVICES.items():
    print(f"── {service}")
    try:
        layers = list_layers(service)
    except Exception as error:
        print(f"   ✗ no se pudo leer el servicio: {error}\n")
        continue
    if wanted_ids is not None:
        layers = [layer for layer in layers if layer["id"] in wanted_ids]
    layers_by_service[service] = layers
    for layer in layers:
        print(f"   [{layer['id']}] {layer['name']}  ({layer['geometry']})")
    print()

── https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR
   [57] Planeamiento General vigente  (esriGeometryPolygon)
   [41] Clasificación  (esriGeometryPolygon)
   [39] Categoría de Suelo Urbanizable  (esriGeometryPolygon)
   [37] Núcleos Rurales  (esriGeometryPolygon)
   [22] Áreas de prevención de riesgos  (esriGeometryPolygon)



## 3. Descargar y guardar

Cada capa se guarda como `<servicio>__<capa>_<fecha>.geojson`. Si el número de elementos descargados
no coincide con el número de identificadores, se avisa: la descarga estaría incompleta.

In [5]:
manifest = {"download_date": TODAY, "base_url": BASE_URL, "layers": []}

for service, layers in layers_by_service.items():
    for layer in layers:
        name = f"{slug(service.split('/')[-1])}__{slug(layer['name'])}"
        path = os.path.join(OUTPUT_DIR, f"{name}_{TODAY}.geojson")
        print(f"↓ {service} / {layer['name']}")
        try:
            collection, expected = download_layer(service, layer["id"])
        except Exception as error:
            print(f"   ✗ falló: {error}\n")
            manifest["layers"].append({"service": service, "layer_id": layer["id"],
                                       "layer_name": layer["name"], "error": str(error)})
            continue
        with open(path, "w", encoding="utf-8") as f:
            json.dump(collection, f, ensure_ascii=False)
        got = len(collection["features"])
        warning = "" if got == expected else f"  ⚠ esperados {expected}"
        print(f"   ✓ {got} elementos → {path}{warning}\n")
        manifest["layers"].append({
            "service": service, "layer_id": layer["id"], "layer_name": layer["name"],
            "url": f"{service_url(service)}/MapServer/{layer['id']}",
            "features": got, "expected": expected, "file": os.path.basename(path),
        })

manifest_path = os.path.join(OUTPUT_DIR, f"manifest_{TODAY}.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("Ficha de la descarga:", manifest_path)

↓ https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR / Planeamiento General vigente
   ✓ 80 elementos → data/raw/visor_rpgur__planeamiento_general_vigente_2026-10-12.geojson

↓ https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR / Clasificación
   ✓ 950 elementos → data/raw/visor_rpgur__clasificacion_2026-10-12.geojson

↓ https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR / Categoría de Suelo Urbanizable
   ✓ 561 elementos → data/raw/visor_rpgur__categoria_de_suelo_urbanizable_2026-10-12.geojson

↓ https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR / Núcleos Rurales
   ✓ 5460 elementos → data/raw/visor_rpgur__nucleos_rurales_2026-10-12.geojson

↓ https://sig.asturias.es/server/rest/services/VisorRPGUR/Visor_RPGUR / Áreas de prevención de riesgos
   ✓ 1153 elementos → data/raw/visor_rpgur__areas_de_prevencion_de_riesgos_2026-10-12.geojson

Ficha de la descarga: data/raw/manifest_2026-10-12.json


## 4. Comprobación rápida

Abre cada fichero descargado con GeoPandas y muestra cuántos elementos tiene, sus columnas y las
dos primeras filas. Así ves qué campos trae cada capa antes de diseñar la limpieza (paso 2).

In [6]:
for entry in manifest["layers"]:
    if "file" not in entry:
        continue
    gdf = gpd.read_file(os.path.join(OUTPUT_DIR, entry["file"]))
    print(f"── {entry['layer_name']}: {len(gdf)} elementos | geometría {gdf.geom_type.unique().tolist()}")
    print("   columnas:", [c for c in gdf.columns if c != "geometry"])
    display(gdf.drop(columns="geometry").head(2))

── Planeamiento General vigente: 80 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['objectid', 'nombre', 'cod_tipo', 'etiqueta', 'instrument', 'ai_bopa', 'ad_bopa', 'tr_bopa', 'id_concejo', 'id_rua', 'st_area(shape)', 'st_length(shape)']


,objectid,nombre,cod_tipo,etiqueta,instrument,ai_bopa,ad_bopa,tr_bopa,id_concejo,id_rua,st_area(shape),st_length(shape)
0,1,Normas Subsidiarias Planeamiento Municipal Yer...,PG,NSPM YERNES Y TAMEZA,NSPM YERNES Y TAMEZA,3.978720e+11,436838400000,NaN,33078,1729,3.053735e+07,26068.584850
1,2,Plan General de Ordenación de Sariego,PG,PGO SARIEGO,PGO SARIEGO,1.139184e+12,1184371200000,1.230595e+12,33065,1893,2.582391e+07,26101.245271


── Clasificación: 950 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['st_area(shape)', 'st_length(shape)', 'objectid', 'clasif', 'uso_global', 'nombre', 'etiqueta', 'id_concejo', 'instrument', 'id_rua']


,st_area(shape),st_length(shape),objectid,clasif,uso_global,nombre,etiqueta,id_concejo,instrument,id_rua
0,78490.721253,2090.940000,1,SU,R,LEVINCO,SU-R-LEVINCO,33002,PGO ALLER,934
1,108921.733916,2375.016838,2,SU,R,COLLANZO,SU-R-COLLANZO,33002,PGO ALLER,934


── Categoría de Suelo Urbanizable: 561 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['objectid', 'cat_sur', 'nombre', 'caracter', 'ord_porm', 'inst_porm', 'uso_pred', 'edif', 'a_medio', 'densidad', 'viv_prot', 'sup_edif_r', 'sup_edif_i', 'sup_edif_c', 'sup_edif_s', 'sup_edif_o', 'sg_vp_inc', 'sg_vp_ads', 'sg_eq_inc', 'sg_eq_ads', 'sg_zv_inc', 'sg_zv_ads', 'sg_el_inc', 'sg_el_ads', 'sist_act', 'plazo', 'plazo_urb', 'plazo_edif', 'etiqueta', 'id_concejo', 'instrument', 'id_rua', 'st_area(shape)', 'st_length(shape)']


,objectid,cat_sur,nombre,caracter,ord_porm,inst_porm,uso_pred,edif,a_medio,densidad,...,sist_act,plazo,plazo_urb,plazo_edif,etiqueta,id_concejo,instrument,id_rua,st_area(shape),st_length(shape)
0,1,SUR-S,SUR-S-SI-01,,,,I,0.5,0.5,0.0,...,SC,0,0,0,SUR-S-SI-01,33001,PGO ALLANDE,2949,34473.405374,1094.273851
1,2,SUR-S,Avenida de Galicia SURS-SR-01,,,,R,0.5,0.5,0.0,...,SC,0,0,0,SUR-S-SR-01,33001,PGO ALLANDE,2949,67726.030751,1538.412859


── Núcleos Rurales: 5460 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['objectid', 'nombre', 'subcat', 'codigocpe', 'etiqueta', 'id_concejo', 'field', 'instrument', 'id_rua', 'st_area(shape)', 'st_length(shape)']


,objectid,nombre,subcat,codigocpe,etiqueta,id_concejo,field,instrument,id_rua,st_area(shape),st_length(shape)
0,1,COLADA,,0,COLADA,33001,None,PGO ALLANDE,2949,23331.392422,778.075909
1,2,VILLANUEVA,,0,VILLANUEVA,33001,None,PGO ALLANDE,2949,72025.991842,1181.850975


── Áreas de prevención de riesgos: 1153 elementos | geometría ['Polygon', 'MultiPolygon']
   columnas: ['objectid', 'numero', 'tipo', 'nombre', 'etiqueta', 'id_concejo', 'instrument', 'id_rua', 'st_area(shape)', 'st_length(shape)']


,objectid,numero,tipo,nombre,etiqueta,id_concejo,instrument,id_rua,st_area(shape),st_length(shape)
0,1,0,TORRENTE,Zona torrencial gen㽻rica,APR-TORRENTE-ZTG,33006,PGO BIMENES,1883,18146.717291,635.259112
1,2,0,TORRENTE,Zona torrencial gen㽻rica,APR-TORRENTE-ZTG,33006,PGO BIMENES,1883,25904.461665,862.593482
